# Notebook 1 — Base Model: Modified EEGNet

**EEG-based Parkinson's Disease Detection using Modified EEGNet**

- **Dataset**: OpenNeuro ds004584 (preprocessed)
- **Input**: Preprocessed EEG epochs shaped `(N, C, T)` where:
  - N = number of epochs
  - C = number of channels (common channels across subjects)
  - T = 500 samples (2s × 250Hz)
- **Output**: Binary classification `PD vs Control`
- **Data Split**: Subject-wise (70% train, 15% val, 15% test)

> **Note**: Data is pre-split by subject in preprocessing to prevent data leakage.

In [ ]:
# ===== Imports =====
import os
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Optional: EEG preprocessing (use if you load raw EEG here)
# import mne

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


In [ ]:
# ===== Config =====
@dataclass
class Config:
    # Data paths (relative to this notebook in models/ folder)
    data_dir: str = "../data/processed/ds002778"
    
    # Model parameters (will be set dynamically after loading data)
    n_channels: int = 63      # Will be updated based on actual data
    n_classes: int = 2        # Binary: PD (1) vs Control (0)
    sampling_rate: int = 250  # From preprocessing
    epoch_seconds: float = 2.0
    n_samples: int = 500      # 2s × 250Hz

    # Model architecture (REDUCED capacity to prevent overfitting)
    F1: int = 4               # Reduced from 8 → 4 temporal filters
    D: int = 2                # Depth multiplier
    F2: int = 8               # Reduced from 16 → 8 pointwise filters
    dropout: float = 0.7      # Increased from 0.6 → 0.7 to reduce overfitting
    kern_len: int = 128        # Larger kernel to capture slower EEG rhythms at 250Hz

    # Training hyperparameters
    batch_size: int = 64
    lr: float = 3e-4           # Reduced from 1e-3 → 3e-4 for more stable training
    weight_decay: float = 1e-3  # Increased from 5e-4 → 1e-3 for stronger regularization
    epochs: int = 20           # Reduced from 40 → 20 (overfitting occurs early)
    patience: int = 5          # Reduced from 10 → 5 (stop sooner)
    
    # Data augmentation (for training only) — ENHANCED
    use_augmentation: bool = True
    noise_std: float = 0.15    # Increased from 0.1 → 0.15
    time_mask_ratio: float = 0.15  # Increased from 0.1 → 0.15
    channel_drop_prob: float = 0.1  # NEW: randomly drop channels
    time_shift_max: int = 25   # NEW: shift signal ±25 samples (100ms @ 250Hz)
    
    # Loss function settings (BALANCED: target 80% PD recall + 70% control recall)
    use_focal_loss: bool = True    # Use Focal Loss to focus on hard examples
    focal_gamma: float = 1.5       # Reduced from 2.0 (less aggressive on hard examples)
    pd_weight_boost: float = 1.8   # Increased from 1.2 → 1.8 to overcome class imbalance
    
    # Threshold tuning settings
    min_pd_recall: float = 0.80    # Target: catch 80% of PD cases
    min_control_recall: float = 0.65  # Target: at least 65% specificity

cfg = Config()

print(f"Model: F1={cfg.F1}, D={cfg.D}, F2={cfg.F2}, dropout={cfg.dropout}, kern_len={cfg.kern_len}")
print(f"LR: {cfg.lr}, Weight decay: {cfg.weight_decay}, Epochs: {cfg.epochs}, Patience: {cfg.patience}")
print(f"Augmentation: noise={cfg.noise_std}, mask={cfg.time_mask_ratio}, ch_drop={cfg.channel_drop_prob}, shift={cfg.time_shift_max}")
print(f"Focal Loss: {cfg.use_focal_loss}, gamma={cfg.focal_gamma}, PD boost={cfg.pd_weight_boost}")
print(f"Target: PD Recall ≥{cfg.min_pd_recall:.0%}, Control Recall ≥{cfg.min_control_recall:.0%}")
print(f"Weight decay: {cfg.weight_decay}, Augmentation: {cfg.use_augmentation}")

## Data Loading

The preprocessing notebook (`preprocessing/preprocess_ds004584.ipynb`) creates:
- `train.npz`, `val.npz`, `test.npz` - Subject-wise split data
- `norm_params.npz` - Normalization parameters
- `channels.json` - Channel names used

Each `.npz` file contains:
- `X`: float32 array `(N, C, T)` - Normalized EEG epochs
- `y`: int labels `(N,)` - 0=Control, 1=PD
- `subject_id`: `(N,)` - Subject ID for each epoch

In [ ]:
# ===== Load preprocessed data =====
def load_split(data_dir: str, split: str):
    """Load a data split (train/val/test) from .npz file."""
    path = Path(data_dir) / f"{split}.npz"
    if not path.exists():
        raise FileNotFoundError(
            f"Expected preprocessed data at {path}.\n"
            "Run preprocessing/preprocess_ds004584.ipynb first to generate the data."
        )
    data = np.load(path, allow_pickle=True)
    X = data["X"].astype(np.float32)  # (N, C, T)
    y = data["y"].astype(np.int64)    # (N,)
    subject_id = data["subject_id"] if "subject_id" in data.files else None
    return X, y, subject_id

# Load all splits
print("Loading preprocessed data...")
X_train, y_train, sid_train = load_split(cfg.data_dir, "train")
X_val, y_val, sid_val = load_split(cfg.data_dir, "val")
X_test, y_test, sid_test = load_split(cfg.data_dir, "test")

# Update config with actual dimensions
cfg.n_channels = X_train.shape[1]
cfg.n_samples = X_train.shape[2]

print(f"\n{'='*50}")
print("DATASET LOADED")
print(f"{'='*50}")
print(f"Train: {X_train.shape} | PD: {(y_train==1).sum()}, Control: {(y_train==0).sum()}")
print(f"Val:   {X_val.shape} | PD: {(y_val==1).sum()}, Control: {(y_val==0).sum()}")
print(f"Test:  {X_test.shape} | PD: {(y_test==1).sum()}, Control: {(y_test==0).sum()}")
print(f"\nChannels: {cfg.n_channels}, Samples: {cfg.n_samples}")

In [ ]:
# ===== EEG Data Augmentation =====
class EEGAugmentation:
    """
    Enhanced EEG-specific data augmentation to reduce overfitting.
    Includes: Gaussian noise, time masking, channel dropout, time shift.
    Applied only during training.
    """
    def __init__(self, noise_std=0.15, time_mask_ratio=0.15, channel_drop_prob=0.1, time_shift_max=25):
        self.noise_std = noise_std
        self.time_mask_ratio = time_mask_ratio
        self.channel_drop_prob = channel_drop_prob
        self.time_shift_max = time_shift_max
    
    def add_gaussian_noise(self, x):
        """Add Gaussian noise to EEG signal."""
        noise = torch.randn_like(x) * self.noise_std
        return x + noise
    
    def time_masking(self, x):
        """Randomly mask a portion of time samples (set to zero)."""
        T = x.shape[-1]
        mask_len = int(T * self.time_mask_ratio)
        if mask_len > 0:
            start = torch.randint(0, T - mask_len, (1,)).item()
            x[..., start:start + mask_len] = 0
        return x
    
    def channel_dropout(self, x):
        """Randomly zero out entire channels to improve spatial robustness."""
        if self.channel_drop_prob <= 0:
            return x
        C = x.shape[-2]
        drop_mask = torch.rand(C, device=x.device) < self.channel_drop_prob
        if drop_mask.any():
            x = x.clone()
            x[drop_mask, :] = 0
        return x
    
    def time_shift(self, x):
        """Randomly shift the signal in time (circular) to improve temporal robustness."""
        if self.time_shift_max <= 0:
            return x
        shift = torch.randint(-self.time_shift_max, self.time_shift_max + 1, (1,)).item()
        return torch.roll(x, shifts=shift, dims=-1)
    
    def __call__(self, x):
        """Apply all augmentations randomly."""
        x = self.add_gaussian_noise(x)
        x = self.time_masking(x)
        x = self.channel_dropout(x)
        x = self.time_shift(x)
        return x


# ===== EEG Dataset =====
class EEGDataset(Dataset):
    """PyTorch Dataset for preprocessed EEG epochs."""
    def __init__(self, X, y, subject_ids=None, augment=False, aug_config=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.subject_ids = subject_ids
        self.augment = augment
        self.augmenter = EEGAugmentation(
            noise_std=aug_config.noise_std if aug_config else 0.15,
            time_mask_ratio=aug_config.time_mask_ratio if aug_config else 0.15,
            channel_drop_prob=aug_config.channel_drop_prob if aug_config else 0.1,
            time_shift_max=aug_config.time_shift_max if aug_config else 25
        ) if augment else None
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment and self.augmenter:
            x = self.augmenter(x)
        return x, self.y[idx]

# Create datasets (with augmentation for training only)
train_dataset = EEGDataset(X_train, y_train, sid_train, 
                           augment=cfg.use_augmentation, aug_config=cfg)
val_dataset = EEGDataset(X_val, y_val, sid_val, augment=False)
test_dataset = EEGDataset(X_test, y_test, sid_test, augment=False)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)} (with augmentation: {cfg.use_augmentation})")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## Model: Modified EEGNet

EEGNet uses **depthwise + separable convolutions** (very lightweight) and is popular for EEG classification.

You can tune:
- temporal kernel length
- depth multiplier
- number of filters
- dropout


In [ ]:
# ===== Modified EEGNet (PyTorch) =====
class SeparableConv2d(nn.Module):
    """Depthwise separable convolution: depthwise -> pointwise."""
    def __init__(self, in_ch, out_ch, kernel_size, padding=(0,0), bias=False):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, kernel_size=kernel_size, 
                                   padding=padding, groups=in_ch, bias=bias)
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=(1,1), bias=bias)
    
    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class EEGNetModified(nn.Module):
    """
    Modified EEGNet for EEG classification (lightweight version).
    
    Architecture:
    1. Temporal convolution (learns frequency filters)
    2. Depthwise convolution (learns spatial filters per channel)
    3. Separable convolution (combines temporal-spatial features)
    4. Classifier head
    
    Args:
        n_channels: Number of EEG channels
        n_classes: Number of output classes
        T: Number of time samples per epoch
        F1: Number of temporal filters (reduced for regularization)
        D: Depth multiplier for depthwise conv
        F2: Number of pointwise filters (reduced for regularization)
        kern_len: Temporal kernel length (related to sampling rate)
        dropout: Dropout probability (increased for regularization)
    """
    def __init__(self, n_channels, n_classes, T, F1=4, D=2, F2=8, kern_len=128, dropout=0.7):
        super().__init__()
        
        # Store config for reference
        self.config = {'F1': F1, 'D': D, 'F2': F2, 'dropout': dropout}
        
        # Block 1: Temporal convolution
        # Input: (B, 1, C, T) -> learns F1 frequency filters
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, kern_len), padding=(0, kern_len//2), bias=False),
            nn.BatchNorm2d(F1)
        )
        
        # Block 2: Depthwise convolution across channels
        # Learns D spatial filters per temporal filter
        self.depthwise_conv = nn.Sequential(
            nn.Conv2d(F1, F1*D, kernel_size=(n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout)
        )
        
        # Block 3: Separable convolution for temporal refinement
        self.separable_conv = nn.Sequential(
            SeparableConv2d(F1*D, F2, kernel_size=(1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout)
        )
        
        # Compute flattened feature dimension
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, T)
            out = self.separable_conv(self.depthwise_conv(self.temporal_conv(dummy)))
            self.feat_dim = out.view(1, -1).shape[1]
        
        # Classifier head
        self.classifier = nn.Linear(self.feat_dim, n_classes)
        
        # Confidence head (for error-correction stage)
        self.conf_head = nn.Linear(self.feat_dim, 1)

    def forward(self, x, return_features=False):
        """
        Forward pass.
        
        Args:
            x: Input tensor (B, C, T)
            return_features: If True, also return intermediate features
            
        Returns:
            logits: Class logits (B, n_classes)
            conf: Confidence score (B, 1) - only if return_features=True
            feats: Flattened features (B, feat_dim) - only if return_features=True
        """
        x = x.unsqueeze(1)  # (B, C, T) -> (B, 1, C, T)
        x = self.temporal_conv(x)
        x = self.depthwise_conv(x)
        x = self.separable_conv(x)
        feats = x.flatten(1)
        
        logits = self.classifier(feats)
        
        if return_features:
            conf = torch.sigmoid(self.conf_head(feats))
            return logits, conf, feats
        return logits


# Create model with reduced capacity
model = EEGNetModified(
    n_channels=cfg.n_channels,
    n_classes=cfg.n_classes,
    T=cfg.n_samples,
    F1=cfg.F1,           # 4 temporal filters
    D=cfg.D,             # Depth multiplier: 2
    F2=cfg.F2,           # 8 pointwise filters
    kern_len=cfg.kern_len,  # 128 (captures slower rhythms)
    dropout=cfg.dropout  # 0.7 (stronger regularization)
).to(DEVICE)

# Print model summary
print(f"Model: EEGNetModified (Lightweight)")
print(f"Architecture: F1={cfg.F1}, D={cfg.D}, F2={cfg.F2}, dropout={cfg.dropout}")
print(f"Input shape: ({cfg.batch_size}, {cfg.n_channels}, {cfg.n_samples})")
print(f"Feature dimension: {model.feat_dim}")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# ===== Training & Evaluation utilities =====
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(1, len(loader))

def eval_model(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            y_true.append(yb.numpy())
            y_pred.append(pred)
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    return acc, f1, cm, y_true, y_pred


# ===== Focal Loss for handling class imbalance =====
class FocalLoss(nn.Module):
    """
    Focal Loss for imbalanced classification.
    Focuses on hard-to-classify examples by down-weighting easy examples.
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        alpha: Class weights (tensor of shape [n_classes])
        gamma: Focusing parameter (higher = more focus on hard examples)
    """
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)  # p_t = probability of correct class
        focal_weight = (1 - pt) ** self.gamma
        
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_weight * ce_loss
        else:
            focal_loss = focal_weight * ce_loss
        
        return focal_loss.mean()


# ===== Class weights to handle imbalance (with PD boost) =====
def compute_class_weights(y, pd_boost=1.0):
    """
    Compute inverse frequency class weights with optional PD boost.
    Higher pd_boost = more emphasis on correctly classifying PD cases.
    """
    classes, counts = np.unique(y, return_counts=True)
    weights = len(y) / (len(classes) * counts)
    # Boost PD class weight to improve recall
    weights[1] *= pd_boost  # PD is class 1
    return torch.tensor(weights, dtype=torch.float32)

class_weights = compute_class_weights(y_train, pd_boost=cfg.pd_weight_boost).to(DEVICE)
print(f"Class weights (with PD boost {cfg.pd_weight_boost}x):")
print(f"  Control={class_weights[0]:.3f}, PD={class_weights[1]:.3f}")

## Training Loop

Training with:
- **Optimizer**: AdamW (lr=3e-4, wd=1e-3)
- **Loss**: Focal Loss (γ=1.5) with PD class weight boost
- **Augmentation**: Noise + Time Mask + Channel Dropout + Time Shift
- **Early stopping**: patience=5 on validation F1
- **Scheduler**: ReduceLROnPlateau (patience=3)

> Hyperparameters tuned to reduce overfitting observed in initial training.

In [ ]:
# ===== Training Loop with Improvements =====
optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# Choose loss function: Focal Loss (better for imbalanced data) or CrossEntropy
if cfg.use_focal_loss:
    criterion = FocalLoss(alpha=class_weights, gamma=cfg.focal_gamma)
    print(f"Using Focal Loss (gamma={cfg.focal_gamma})")
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print("Using CrossEntropyLoss")

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

# Training history
history = {
    'train_loss': [], 'train_acc': [], 'train_f1': [],
    'val_loss': [], 'val_acc': [], 'val_f1': [], 'lr': []
}

best_val_f1 = -1
best_model_state = None
best_epoch = 0  # Track best epoch explicitly
patience_counter = 0

print("=" * 60)
print("TRAINING STARTED")
print(f"Improvements: Dropout={cfg.dropout}, F1={cfg.F1}, F2={cfg.F2}, kern_len={cfg.kern_len}")
print(f"LR={cfg.lr}, Weight decay={cfg.weight_decay}, Epochs={cfg.epochs}, Patience={cfg.patience}")
print(f"Augmentation: noise={cfg.noise_std}, mask={cfg.time_mask_ratio}, ch_drop={cfg.channel_drop_prob}, shift={cfg.time_shift_max}")
print(f"Focal Loss: {cfg.use_focal_loss}, PD Weight Boost: {cfg.pd_weight_boost}")
print("=" * 60)

for epoch in range(1, cfg.epochs + 1):
    # === Training ===
    model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []
    
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        train_labels.extend(yb.cpu().numpy())
    
    train_loss /= len(train_loader)
    train_acc = accuracy_score(train_labels, train_preds)
    train_f1 = f1_score(train_labels, train_preds)
    
    # === Validation ===
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []
    
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            
            val_loss += loss.item()
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_labels.extend(yb.cpu().numpy())
    
    val_loss /= len(val_loader)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds)
    
    # Get current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    
    # Update learning rate scheduler
    scheduler.step(val_f1)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)
    
    # Check for best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch  # Log best epoch explicitly
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        marker = " *** BEST ***"
    else:
        patience_counter += 1
        marker = ""
    
    # Print progress
    print(f"Epoch {epoch:03d} | LR: {current_lr:.2e} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f} | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}{marker}")
    
    # Early stopping
    if patience_counter >= cfg.patience:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {cfg.patience} epochs)")
        break

print(f"\n{'='*60}")
print(f"Training complete! Best Val F1: {best_val_f1:.4f} at Epoch {best_epoch}")
print(f"{'='*60}")

## Evaluation on Test Set

Load the best model and evaluate on the held-out test set.

In [ ]:
# ===== Threshold Tuning on Validation Set =====
def find_optimal_threshold(model, loader, min_pd_recall=0.80, min_control_recall=0.65):
    """
    Find optimal classification threshold that balances PD recall and specificity.
    
    Args:
        model: Trained model
        loader: Validation data loader
        min_pd_recall: Minimum PD recall (sensitivity) target
        min_control_recall: Minimum control recall (specificity) target
    """
    model.eval()
    all_probs, all_labels = [], []
    
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)[:, 1]  # P(PD)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(yb.numpy())
    
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    # Search for optimal threshold that balances both recalls
    results = []
    for thresh in np.arange(0.20, 0.70, 0.01):
        preds = (all_probs >= thresh).astype(int)
        
        # Calculate metrics
        pd_mask = all_labels == 1
        control_mask = all_labels == 0
        
        pd_recall = np.sum(preds[pd_mask] == 1) / np.sum(pd_mask) if np.sum(pd_mask) > 0 else 0
        control_recall = np.sum(preds[control_mask] == 0) / np.sum(control_mask) if np.sum(control_mask) > 0 else 0
        f1 = f1_score(all_labels, preds)
        
        # Balanced accuracy (average of both recalls)
        balanced_acc = (pd_recall + control_recall) / 2
        
        results.append({
            'threshold': thresh,
            'pd_recall': pd_recall,
            'control_recall': control_recall,
            'f1': f1,
            'balanced_acc': balanced_acc
        })
    
    # Find thresholds meeting both constraints
    valid_results = [r for r in results 
                     if r['pd_recall'] >= min_pd_recall and r['control_recall'] >= min_control_recall]
    
    if valid_results:
        # Among valid thresholds, pick the one with best balanced accuracy
        best = max(valid_results, key=lambda x: x['balanced_acc'])
        print(f"✅ Found threshold meeting both constraints!")
    else:
        # Fallback: find best trade-off using weighted score
        print(f"⚠️ No threshold meets both constraints. Finding best trade-off...")
        for r in results:
            # Weighted score: favor PD recall slightly more (clinical importance)
            r['score'] = 0.5 * r['pd_recall'] + 0.3 * r['control_recall'] + 0.2 * r['f1']
        best = max(results, key=lambda x: x['score'])
    
    return best['threshold'], best['f1'], best['pd_recall'], best['control_recall']

# Load best model and find optimal threshold
model.load_state_dict(best_model_state)
model.eval()

optimal_threshold, val_f1, val_pd_recall, val_control_recall = find_optimal_threshold(
    model, val_loader, 
    min_pd_recall=cfg.min_pd_recall, 
    min_control_recall=cfg.min_control_recall
)
print(f"\nOptimal threshold: {optimal_threshold:.2f}")
print(f"Val F1: {val_f1:.4f}")
print(f"Val PD Recall: {val_pd_recall:.4f} (target ≥{cfg.min_pd_recall:.0%})")
print(f"Val Control Recall: {val_control_recall:.4f} (target ≥{cfg.min_control_recall:.0%})")

In [ ]:
# ===== Window-Level and Subject-Level Evaluation =====

def evaluate_with_threshold(model, loader, threshold=0.5):
    """Evaluate model with custom threshold."""
    model.eval()
    all_probs, all_labels = [], []
    
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(yb.numpy())
    
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds = (all_probs >= threshold).astype(int)
    
    return all_labels, preds, all_probs


def subject_level_evaluation(labels, preds, probs, subject_ids):
    """Aggregate window predictions to subject-level using majority voting."""
    if subject_ids is None:
        return None, None, None
    
    unique_subjects = np.unique(subject_ids)
    subj_labels, subj_preds, subj_probs = [], [], []
    
    for sid in unique_subjects:
        mask = subject_ids == sid
        # Subject label (should be same for all windows from same subject)
        subj_label = labels[mask][0]
        # Majority voting for prediction
        subj_pred = int(np.mean(preds[mask]) >= 0.5)
        # Average probability
        subj_prob = np.mean(probs[mask])
        
        subj_labels.append(subj_label)
        subj_preds.append(subj_pred)
        subj_probs.append(subj_prob)
    
    return np.array(subj_labels), np.array(subj_preds), np.array(subj_probs)


def print_confusion_matrix(cm, title="Confusion Matrix"):
    """Print a nicely formatted confusion matrix using ASCII characters."""
    tn, fp, fn, tp = cm.ravel()
    print(f"\n{title}:")
    print("+---------------+---------------+---------------+")
    print("|               |          Predicted            |")
    print("|               +---------------+---------------+")
    print("|               |   Control     |      PD       |")
    print("+---------------+---------------+---------------+")
    print(f"| Actual Ctrl   |    {tn:5d}      |    {fp:5d}      |")
    print(f"| Actual PD     |    {fn:5d}      |    {tp:5d}      |")
    print("+---------------+---------------+---------------+")
    print(f"\nTN={tn}, FP={fp}, FN={fn}, TP={tp}")


def print_detailed_metrics(labels, preds, level="Window"):
    """Print detailed metrics including recall for both classes."""
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    cm = confusion_matrix(labels, preds)
    
    # Calculate per-class metrics
    tn, fp, fn, tp = cm.ravel()
    control_recall = tn / (tn + fp) if (tn + fp) > 0 else 0  # Specificity
    pd_recall = tp / (tp + fn) if (tp + fn) > 0 else 0       # Sensitivity
    pd_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    print(f"\n{level}-Level Metrics:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 Score (PD): {f1:.4f}")
    print(f"  PD Recall (Sensitivity): {pd_recall:.4f} {'[OK]' if pd_recall >= 0.75 else '[LOW]'}")
    print(f"  PD Precision: {pd_precision:.4f}")
    print(f"  Control Recall (Specificity): {control_recall:.4f}")
    print(f"  False Positives: {fp}, False Negatives: {fn}")
    
    return acc, f1, cm, pd_recall


# === Window-Level Evaluation ===
print("=" * 60)
print("TEST SET EVALUATION - WINDOW LEVEL")
print("=" * 60)

# With default threshold (0.5)
test_labels_default, test_preds_default, test_probs = evaluate_with_threshold(
    model, test_loader, threshold=0.5)
print("\n[Threshold = 0.50 (default)]")
print_detailed_metrics(test_labels_default, test_preds_default, "Window")

# With optimized threshold (recall-aware)
test_labels, test_preds, _ = evaluate_with_threshold(
    model, test_loader, threshold=optimal_threshold)
print(f"\n[Threshold = {optimal_threshold:.2f} (optimized with recall constraint)]")
test_acc, test_f1, test_cm, test_pd_recall = print_detailed_metrics(test_labels, test_preds, "Window")

print_confusion_matrix(test_cm, "Window-Level Confusion Matrix")

print(f"\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=['Control', 'PD'], digits=4))


# === Subject-Level Evaluation ===
print("\n" + "=" * 60)
print("TEST SET EVALUATION - SUBJECT LEVEL")
print("=" * 60)

subj_labels, subj_preds, subj_probs = subject_level_evaluation(
    test_labels, test_preds, test_probs, sid_test)

if subj_labels is not None:
    print(f"\nNumber of test subjects: {len(subj_labels)}")
    subj_acc, subj_f1, subj_cm, subj_pd_recall = print_detailed_metrics(subj_labels, subj_preds, "Subject")
    
    print_confusion_matrix(subj_cm, "Subject-Level Confusion Matrix")
    
    print(f"\nSubject-Level Classification Report:")
    print(classification_report(subj_labels, subj_preds, target_names=['Control', 'PD'], digits=4))
else:
    subj_acc, subj_f1, subj_cm = None, None, None
    print("\nSubject IDs not available in test data. Skipping subject-level evaluation.")

## Training History Visualization

In [ ]:
# ===== Plot Training History =====
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss over Epochs')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy over Epochs')
axes[1].legend()
axes[1].grid(True)

# F1 Score
axes[2].plot(history['train_f1'], label='Train')
axes[2].plot(history['val_f1'], label='Validation')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('F1 Score')
axes[2].set_title('F1 Score over Epochs')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

## Detailed Evaluation Visualizations

Confusion matrix heatmaps, ROC curve, probability distributions, and final results summary.

In [ ]:
# ===== Confusion Matrix Heatmaps (Window + Subject Level) =====
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Window-Level Confusion Matrix ---
sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'],
            annot_kws={'size': 16, 'weight': 'bold'})
axes[0].set_xlabel('Predicted', fontsize=13)
axes[0].set_ylabel('Actual', fontsize=13)
axes[0].set_title(f'Window-Level Confusion Matrix\n(threshold={optimal_threshold:.2f})', fontsize=14, weight='bold')

# Add percentage annotations
tn, fp, fn, tp = test_cm.ravel()
total = tn + fp + fn + tp
for i, (val, pct) in enumerate(zip([tn, fp, fn, tp],
                                     [tn/total, fp/total, fn/total, tp/total])):
    row, col = divmod(i, 2)
    axes[0].text(col + 0.5, row + 0.72, f'({pct:.1%})', ha='center', va='center',
                 fontsize=10, color='gray')

# --- Subject-Level Confusion Matrix ---
if subj_cm is not None:
    sns.heatmap(subj_cm, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
                xticklabels=['Control', 'PD'], yticklabels=['Control', 'PD'],
                annot_kws={'size': 16, 'weight': 'bold'})
    axes[1].set_xlabel('Predicted', fontsize=13)
    axes[1].set_ylabel('Actual', fontsize=13)
    axes[1].set_title('Subject-Level Confusion Matrix\n(majority voting)', fontsize=14, weight='bold')
    
    s_tn, s_fp, s_fn, s_tp = subj_cm.ravel()
    s_total = s_tn + s_fp + s_fn + s_tp
    for i, (val, pct) in enumerate(zip([s_tn, s_fp, s_fn, s_tp],
                                         [s_tn/s_total, s_fp/s_total, s_fn/s_total, s_tp/s_total])):
        row, col = divmod(i, 2)
        axes[1].text(col + 0.5, row + 0.72, f'({pct:.1%})', ha='center', va='center',
                     fontsize=10, color='gray')
else:
    axes[1].text(0.5, 0.5, 'Subject IDs not available', ha='center', va='center', fontsize=14)
    axes[1].set_title('Subject-Level Confusion Matrix', fontsize=14, weight='bold')

plt.tight_layout()
plt.savefig('../training/saved_models/eegnet_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print("Confusion matrices saved.")

In [ ]:
# ===== ROC Curve & AUC =====
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- ROC Curve ---
fpr, tpr, thresholds_roc = roc_curve(test_labels, test_probs)
roc_auc = auc(fpr, tpr)

axes[0].plot(fpr, tpr, color='#2196F3', lw=2.5, label=f'EEGNet Modified (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.50)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#2196F3')

# Mark optimal threshold point
opt_idx = np.argmin(np.abs(thresholds_roc - optimal_threshold))
axes[0].scatter(fpr[opt_idx], tpr[opt_idx], color='red', s=100, zorder=5,
                label=f'Threshold={optimal_threshold:.2f}')

axes[0].set_xlabel('False Positive Rate', fontsize=13)
axes[0].set_ylabel('True Positive Rate', fontsize=13)
axes[0].set_title('ROC Curve', fontsize=14, weight='bold')
axes[0].legend(loc='lower right', fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([-0.02, 1.02])
axes[0].set_ylim([-0.02, 1.02])

# --- Precision-Recall Curve ---
precision, recall, thresholds_pr = precision_recall_curve(test_labels, test_probs)
ap = average_precision_score(test_labels, test_probs)

axes[1].plot(recall, precision, color='#FF5722', lw=2.5, label=f'EEGNet Modified (AP = {ap:.4f})')
axes[1].fill_between(recall, precision, alpha=0.1, color='#FF5722')
axes[1].axhline(y=(test_labels == 1).sum() / len(test_labels), color='k', ls='--', lw=1,
                alpha=0.5, label='Baseline (prevalence)')

axes[1].set_xlabel('Recall', fontsize=13)
axes[1].set_ylabel('Precision', fontsize=13)
axes[1].set_title('Precision-Recall Curve', fontsize=14, weight='bold')
axes[1].legend(loc='lower left', fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([-0.02, 1.02])
axes[1].set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig('../training/saved_models/eegnet_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"ROC AUC: {roc_auc:.4f}")
print(f"Average Precision: {ap:.4f}")

In [ ]:
# ===== Prediction Probability Distribution =====
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Histogram of P(PD) by true class ---
control_probs = test_probs[test_labels == 0]
pd_probs = test_probs[test_labels == 1]

axes[0].hist(control_probs, bins=40, alpha=0.7, color='#4CAF50', label=f'Control (n={len(control_probs)})',
             edgecolor='white', density=True)
axes[0].hist(pd_probs, bins=40, alpha=0.7, color='#F44336', label=f'PD (n={len(pd_probs)})',
             edgecolor='white', density=True)
axes[0].axvline(x=optimal_threshold, color='black', ls='--', lw=2,
                label=f'Threshold = {optimal_threshold:.2f}')
axes[0].axvline(x=0.5, color='gray', ls=':', lw=1.5, label='Default = 0.50')
axes[0].set_xlabel('P(PD)', fontsize=13)
axes[0].set_ylabel('Density', fontsize=13)
axes[0].set_title('Prediction Probability Distribution', fontsize=14, weight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# --- Box plot of probabilities by class ---
box_data = [control_probs, pd_probs]
bp = axes[1].boxplot(box_data, labels=['Control', 'PD'], patch_artist=True, widths=0.5,
                     medianprops=dict(color='black', lw=2))
colors = ['#4CAF50', '#F44336']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].axhline(y=optimal_threshold, color='black', ls='--', lw=2,
                label=f'Threshold = {optimal_threshold:.2f}')
axes[1].set_ylabel('P(PD)', fontsize=13)
axes[1].set_title('Probability Distribution by True Class', fontsize=14, weight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../training/saved_models/eegnet_probability_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Control probs — mean: {control_probs.mean():.4f}, median: {np.median(control_probs):.4f}")
print(f"PD probs      — mean: {pd_probs.mean():.4f}, median: {np.median(pd_probs):.4f}")

In [ ]:
# ===== Final Results Summary =====
from sklearn.metrics import roc_curve, auc

fpr_, tpr_, _ = roc_curve(test_labels, test_probs)
roc_auc_ = auc(fpr_, tpr_)
tn, fp, fn, tp = test_cm.ravel()

print("=" * 70)
print("       MODIFIED EEGNet — FINAL RESULTS SUMMARY")
print("=" * 70)

print(f"\n{'MODEL ARCHITECTURE':}")
print(f"  Model:              Modified EEGNet (Lightweight)")
print(f"  Parameters:         {sum(p.numel() for p in model.parameters()):,}")
print(f"  Architecture:       F1={cfg.F1}, D={cfg.D}, F2={cfg.F2}")
print(f"  Dropout:            {cfg.dropout}")
print(f"  Feature dimension:  {model.feat_dim}")

print(f"\n{'TRAINING CONFIGURATION':}")
print(f"  Optimizer:          AdamW (lr={cfg.lr}, wd={cfg.weight_decay})")
print(f"  Loss:               {'Focal Loss (γ=' + str(cfg.focal_gamma) + ')' if cfg.use_focal_loss else 'CrossEntropy'}")
print(f"  Augmentation:       {'Noise + Time Mask' if cfg.use_augmentation else 'None'}")
print(f"  Best epoch:         {best_epoch}")
print(f"  Optimal threshold:  {optimal_threshold:.2f}")

print(f"\n{'DATASET':}")
print(f"  Train:  {X_train.shape[0]:,} epochs ({(y_train==1).sum()} PD, {(y_train==0).sum()} Ctrl)")
print(f"  Val:    {X_val.shape[0]:,} epochs ({(y_val==1).sum()} PD, {(y_val==0).sum()} Ctrl)")
print(f"  Test:   {X_test.shape[0]:,} epochs ({(y_test==1).sum()} PD, {(y_test==0).sum()} Ctrl)")
print(f"  Input:  ({cfg.n_channels} channels, {cfg.n_samples} samples @ {cfg.sampling_rate}Hz)")

print(f"\n{'TEST RESULTS — WINDOW LEVEL':}")
print(f"  Accuracy:           {test_acc:.4f}")
print(f"  F1 Score (PD):      {test_f1:.4f}")
print(f"  ROC AUC:            {roc_auc_:.4f}")
print(f"  Sensitivity (PD):   {tp/(tp+fn):.4f}  (TP={tp}, FN={fn})")
print(f"  Specificity (Ctrl): {tn/(tn+fp):.4f}  (TN={tn}, FP={fp})")

if subj_cm is not None:
    s_tn, s_fp, s_fn, s_tp = subj_cm.ravel()
    print(f"\n{'TEST RESULTS — SUBJECT LEVEL':}")
    print(f"  Subjects tested:    {len(subj_labels)}")
    print(f"  Accuracy:           {subj_acc:.4f}")
    print(f"  F1 Score (PD):      {subj_f1:.4f}")
    print(f"  Sensitivity (PD):   {s_tp/(s_tp+s_fn):.4f}  ({s_tp}/{s_tp+s_fn} PD correctly identified)")
    print(f"  Specificity (Ctrl): {s_tn/(s_tn+s_fp):.4f}  ({s_tn}/{s_tn+s_fp} Ctrl correctly identified)")
    print(f"  Misclassified:      {s_fp} FP + {s_fn} FN = {s_fp+s_fn} out of {len(subj_labels)}")

print(f"\n{'=' * 70}")
print("  Key Takeaway: EEGNet achieves strong subject-level accuracy")
print("  with only ~1.3K parameters — highly suitable for edge deployment.")
print("=" * 70)

## Save Model

Save the trained model for later use or error-correction stage.

In [ ]:
# ===== Save Model =====
save_dir = Path("../training/saved_models")
save_dir.mkdir(parents=True, exist_ok=True)

model_path = save_dir / "eegnet_modified_parkinsons.pth"
torch.save({
    'model_state_dict': best_model_state,
    'config': {
        'n_channels': cfg.n_channels,
        'n_classes': cfg.n_classes,
        'n_samples': cfg.n_samples,
        'F1': cfg.F1,
        'D': cfg.D,
        'F2': cfg.F2,
        'dropout': cfg.dropout,
    },
    'training_config': {
        'lr': cfg.lr,
        'weight_decay': cfg.weight_decay,
        'batch_size': cfg.batch_size,
        'augmentation': cfg.use_augmentation,
    },
    'best_epoch': best_epoch,
    'optimal_threshold': optimal_threshold,
    'test_metrics': {
        'window_level': {
            'accuracy': test_acc,
            'f1_score': test_f1,
            'confusion_matrix': test_cm.tolist(),
            'threshold': optimal_threshold,
        },
        'subject_level': {
            'accuracy': subj_acc if subj_labels is not None else None,
            'f1_score': subj_f1 if subj_labels is not None else None,
            'confusion_matrix': subj_cm.tolist() if subj_labels is not None else None,
        },
    },
    'history': history,
}, model_path)

print(f"Model saved to: {model_path}")
print(f"\nBest Epoch: {best_epoch}")
print(f"Optimal Threshold: {optimal_threshold:.2f}")
print(f"\nWindow-Level: Acc={test_acc:.4f}, F1={test_f1:.4f}")
if subj_labels is not None:
    print(f"Subject-Level: Acc={subj_acc:.4f}, F1={subj_f1:.4f}")

In [ ]:
print(model)